# Applio Darwin — Notebook de production

Ce notebook est conçu pour un runtime Colab/Kaggle jetable : le dépôt GitHub contient le code, Google Drive conserve les fichiers lourds, et le runtime est reconstruit automatiquement à chaque session.

Après la création du dépôt, la seule valeur à renseigner est `GITHUB_REPO` ci-dessous. Ensuite, les sessions suivantes récupèrent automatiquement la dernière version du dépôt.


## 1. Récupérer le projet depuis GitHub

In [ ]:
GITHUB_REPO = "https://github.com/TON_USERNAME/applio-darwin.git"  # @param {type:"string"}

import subprocess
from pathlib import Path

project_dir = Path("/content/applio-darwin")

if not GITHUB_REPO or "TON_USERNAME" in GITHUB_REPO:
    raise ValueError("Renseigne GITHUB_REPO avec l'URL réelle de ton dépôt GitHub avant de lancer cette cellule.")

if not project_dir.exists():
    subprocess.run(["git", "clone", GITHUB_REPO, str(project_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)

%cd /content/applio-darwin
print("✅ Projet récupéré depuis GitHub.")


## 2. Connecter Google Drive

In [ ]:
from google.colab import drive
from google.colab._message import MessageError

try:
    drive.mount("/content/drive")
    print("✅ Drive monté.")
except MessageError:
    print("❌ Échec du montage de Drive — réessaie cette cellule.")


## 3. Préparer automatiquement la configuration

Le dépôt versionne uniquement `config.example.json`. Cette cellule crée automatiquement la configuration locale si nécessaire. Elle ne demande aucune modification manuelle à chaque session.


In [ ]:
from pathlib import Path
import shutil

example = Path("config/config.example.json")
target = Path("config/config.json")

if not example.exists():
    raise FileNotFoundError(f"Modèle de configuration introuvable : {example}")

if not target.exists():
    shutil.copy2(example, target)
    print("✅ config/config.json créé automatiquement.")
else:
    print("✅ config/config.json déjà présent.")

print(target.read_text(encoding="utf-8"))


## 4. Installer automatiquement Applio

Cette étape reconstruit l’environnement nécessaire à chaque nouvelle session.


In [ ]:
!python scripts/setup.py --config config/config.json


## 5. Diagnostic complet

Le diagnostic vérifie l’installation, les ressources, le GPU, le modèle Darwin et l’audio avant l’inférence.


In [ ]:
!python scripts/check_environment.py --config config/config.json


## 6. Inférence — Niveau 1

La conversion utilise automatiquement le modèle `darwin` conservé sur Google Drive et sauvegarde le résultat et le log dans `ApplioExported/`.


In [ ]:
!python scripts/inference.py --config config/config.json


In [ ]:
# Écoute directement le résultat ici
from IPython.display import Audio, display
import json

with open("config/config.json") as f:
    cfg = json.load(f)

output_path = f"{cfg['drive']['root']}/{cfg['drive']['export_folder']}/{cfg['model_name']}_output.wav"
display(Audio(output_path))


---
## Section Entraînement — Niveau 3

⚠️ **Ne pas exécuter avant d'avoir validé les Niveaux 1 et 2.**
Ces cellules créent/améliorent le modèle `darwin` à partir d'un dataset
vocal placé dans `Drive/Darwin_Dataset/`. Chaque étape s'exécute
séparément et dans l'ordre : preprocess → extract → index → train.

In [ ]:
!python scripts/train.py --config config/config.json --step preprocess


In [ ]:
!python scripts/train.py --config config/config.json --step extract


In [ ]:
!python scripts/train.py --config config/config.json --step index


In [ ]:
!python scripts/train.py --config config/config.json --step train
